# 🌍 Global Child Health Data Extraction Tutorial

**A Beginner's Guide to Fetching Health Indicators from Global APIs**

---

Welcome! This notebook will teach you how to extract child health data from major global health databases. By the end, you'll have a consolidated dataset ready for analysis.

## What You'll Learn

1. 📦 **Setting up** your Python environment
2. 🏦 **World Bank API** – Fetch malnutrition, birth weight, and mortality data
3. 🏥 **WHO GHO API** – Get wasting prevalence and breastfeeding statistics
4. 👶 **UNICEF SDMX API** – Retrieve under-5 population counts
5. 🔄 **Processing & Consolidating** – Clean and combine all data sources
6. 💾 **Saving Results** – Export to Excel/CSV for further use

---


## 📦 Step 1: Install Required Packages

First, let's make sure we have all the libraries we need. Run the cell below to install them.


In [2]:
# Install required packages (run this once)
%pip install pandas requests openpyxl country_converter tqdm --quiet

print("✅ All packages installed!")



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ All packages installed!


## 📚 Step 2: Import Libraries

Now let's import the libraries we'll use throughout this tutorial.


In [3]:
import pandas as pd
import requests
import time
import numpy as np
from datetime import datetime

# For converting country codes to names
import country_converter as coco

# For progress bars
from tqdm.notebook import tqdm

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print(f"🚀 Tutorial started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("✅ Libraries loaded successfully!")


🚀 Tutorial started: 2026-01-07 15:33:14
✅ Libraries loaded successfully!


---

## 🏦 Step 3: Fetching Data from the World Bank API

The [World Bank Open Data](https://data.worldbank.org/) provides free access to global development data. We'll fetch several child health indicators.

### How the World Bank API Works

The API uses **indicator codes** to identify specific datasets. For example:
- `SH.STA.MALN.ZS` = Malnutrition prevalence (weight-for-age < -2 SD)
- `SH.STA.BRTW.ZS` = Low birth weight prevalence
- `SH.DYN.MORT` = Under-5 mortality rate

The base URL format is:
```
http://api.worldbank.org/v2/country/all/indicator/{INDICATOR_CODE}?format=json
```


In [4]:
def fetch_world_bank_indicator(indicator_id, indicator_name):
    """
    Fetches data for a single indicator from the World Bank API.
    
    Parameters:
    -----------
    indicator_id : str
        The World Bank indicator code (e.g., 'SH.STA.MALN.ZS')
    indicator_name : str
        A human-readable name for the indicator
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: country, year, value
    """
    print(f"📥 Fetching: {indicator_name}...")
    
    all_data = []
    page = 1
    
    while True:
        # Build the API URL with pagination
        url = f"http://api.worldbank.org/v2/country/all/indicator/{indicator_id}?format=json&page={page}&per_page=1000"
        
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()  # Raise error for bad status codes
            data = response.json()
            
            # World Bank returns [metadata, data] - we want data[1]
            if not data or len(data) < 2 or not data[1]:
                break  # No more data
            
            all_data.extend(data[1])
            page += 1
            time.sleep(0.3)  # Be nice to the API - don't hammer it!
            
        except Exception as e:
            print(f"   ⚠️ Error on page {page}: {e}")
            break
    
    if not all_data:
        print(f"   ❌ No data found for {indicator_name}")
        return pd.DataFrame()
    
    # Convert to DataFrame
    df = pd.DataFrame(all_data)
    
    # Extract country name from nested dict
    df['country'] = df['country'].apply(lambda x: x['value'] if isinstance(x, dict) else x)
    
    # Keep only rows with actual values
    df = df[['country', 'date', 'value']].dropna(subset=['value'])
    df.columns = ['country', 'year', 'value']
    
    # Convert to numeric
    df['year'] = pd.to_numeric(df['year'])
    df['value'] = pd.to_numeric(df['value'])
    
    print(f"   ✅ Found {len(df):,} records")
    return df


### Let's Try It!

Let's fetch malnutrition data and see what we get:


In [5]:
# Fetch malnutrition data
malnutrition_df = fetch_world_bank_indicator(
    indicator_id='SH.STA.MALN.ZS',
    indicator_name='Malnutrition (weight-for-age < -2 SD)'
)

# Let's peek at the data!
print("\n📊 Sample of the data:")
malnutrition_df.head(10)


📥 Fetching: Malnutrition (weight-for-age < -2 SD)...
   ✅ Found 1,477 records

📊 Sample of the data:


,country,year,value
390,East Asia & Pacific,2024,5.42
391,East Asia & Pacific,2023,5.41
392,East Asia & Pacific,2022,5.38
393,East Asia & Pacific,2021,5.36
394,East Asia & Pacific,2020,5.41
395,East Asia & Pacific,2019,5.52
396,East Asia & Pacific,2018,5.67
397,East Asia & Pacific,2017,5.87
398,East Asia & Pacific,2016,6.10
399,East Asia & Pacific,2015,6.34


In [6]:
# Define all World Bank indicators we want
wb_indicators = {
    'SH.STA.MALN.ZS': 'Malnutrition (weight-for-age < -2 SD)',
    'SH.STA.BRTW.ZS': 'Low birth weight (≤2500g)',
    'EG.USE.COMM.CL.ZS': 'Solid fuel use (%)',
    'SH.DYN.MORT': 'Under-5 mortality rate (per 1000)',
}

# Fetch all indicators
world_bank_data = []

for indicator_id, indicator_name in wb_indicators.items():
    df = fetch_world_bank_indicator(indicator_id, indicator_name)
    if not df.empty:
        df['indicator'] = indicator_name
        df['source'] = 'World Bank'
        world_bank_data.append(df)

# Combine all World Bank data
wb_combined = pd.concat(world_bank_data, ignore_index=True)
print(f"\n✅ Total World Bank records: {len(wb_combined):,}")


📥 Fetching: Malnutrition (weight-for-age < -2 SD)...
   ✅ Found 1,477 records
📥 Fetching: Low birth weight (≤2500g)...
   ✅ Found 3,953 records
📥 Fetching: Solid fuel use (%)...
   ✅ Found 6,521 records
📥 Fetching: Under-5 mortality rate (per 1000)...
   ✅ Found 13,251 records

✅ Total World Bank records: 25,202


---

## 🏥 Step 4: Fetching Data from the WHO GHO API

The [WHO Global Health Observatory](https://www.who.int/data/gho) provides health statistics for all WHO member states.

### How the WHO GHO API Works

The WHO uses an OData-style API. The base URL is:
```
https://ghoapi.azureedge.net/api/{INDICATOR_CODE}
```

Key indicators we'll use:
- `NUTRITION_WH_2` = Wasting prevalence in children under 5
- `WHOSIS_000006` = Exclusive breastfeeding under 6 months


In [7]:
def fetch_who_indicator(indicator_code, indicator_name):
    """
    Fetches data from the WHO GHO API.
    
    Parameters:
    -----------
    indicator_code : str
        The WHO indicator code (e.g., 'NUTRITION_WH_2')
    indicator_name : str
        A human-readable name for the indicator
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: country, year, value
    """
    print(f"📥 Fetching: {indicator_name}...")
    
    url = f"https://ghoapi.azureedge.net/api/{indicator_code}"
    
    try:
        response = requests.get(url, timeout=45)
        response.raise_for_status()
        data = response.json().get('value', [])
        
        if not data:
            print(f"   ❌ No data found")
            return pd.DataFrame()
        
        df = pd.DataFrame(data)
        
        # Filter to country-level data only (not regions)
        df = df[df['SpatialDimType'] == 'COUNTRY']
        
        # Select and rename columns
        df = df[['SpatialDim', 'TimeDim', 'NumericValue']]
        df.columns = ['country', 'year', 'value']
        
        # Convert ISO3 codes to country names
        df['country'] = coco.convert(df['country'].tolist(), to='name_short', not_found=np.nan)
        df = df.dropna(subset=['country'])
        
        # Convert to numeric
        df['year'] = pd.to_numeric(df['year'])
        df['value'] = pd.to_numeric(df['value'])
        
        print(f"   ✅ Found {len(df):,} records")
        return df
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return pd.DataFrame()

# Fetch WHO indicators
who_indicators = {
    'NUTRITION_WH_2': 'Wasting prevalence (children under 5)',
    'WHOSIS_000006': 'Exclusive breastfeeding under 6 months (%)',
}

who_data = []
for indicator_code, indicator_name in who_indicators.items():
    df = fetch_who_indicator(indicator_code, indicator_name)
    if not df.empty:
        df['indicator'] = indicator_name
        df['source'] = 'WHO'
        who_data.append(df)

who_combined = pd.concat(who_data, ignore_index=True) if who_data else pd.DataFrame()
print(f"\n✅ Total WHO records: {len(who_combined):,}")


📥 Fetching: Wasting prevalence (children under 5)...
   ✅ Found 67,739 records
📥 Fetching: Exclusive breastfeeding under 6 months (%)...
   ✅ Found 607 records

✅ Total WHO records: 68,346


---

## 👶 Step 5: Fetching Population Data from UNICEF

The [UNICEF Data Warehouse](https://data.unicef.org/) provides demographic data including under-5 population counts.

### How the UNICEF SDMX API Works

UNICEF uses the SDMX (Statistical Data and Metadata eXchange) format. The response structure is a bit complex, but we'll walk through it step by step.


In [8]:
def fetch_unicef_under5_population():
    """
    Fetches under-5 population data from UNICEF SDMX API.
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: country, year, value (population count)
    """
    print("📥 Fetching: Under-5 population from UNICEF...")
    
    url = "https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,DM,1.0/.DM_POP_U5?format=sdmx-json"
    
    try:
        response = requests.get(url, timeout=90)
        response.raise_for_status()
        payload = response.json()
        
        # Build lookup tables from the SDMX structure
        series_dims = payload["data"]["structure"]["dimensions"]["series"]
        ref_areas = series_dims[0]["values"]  # Countries (ISO3 codes)
        ref_lookup = {str(idx): v["id"] for idx, v in enumerate(ref_areas)}
        
        time_values = payload["data"]["structure"]["dimensions"]["observation"][0]["values"]
        time_lookup = {str(idx): int(v["id"]) for idx, v in enumerate(time_values)}
        
        # Extract the actual data
        series_dict = payload["data"]["dataSets"][0]["series"]
        
        records = []
        for series_key, series_val in series_dict.items():
            ref_idx = series_key.split(":")[0]
            iso3 = ref_lookup.get(ref_idx)
            if not iso3:
                continue
                
            for time_idx, obs in series_val["observations"].items():
                year = time_lookup.get(time_idx)
                if year is None:
                    continue
                try:
                    # UNICEF returns population in thousands!
                    value = float(obs[0]) * 1000
                    records.append({"country": iso3, "year": year, "value": value})
                except (TypeError, ValueError):
                    continue
        
        if not records:
            print("   ❌ No records parsed")
            return pd.DataFrame()
        
        df = pd.DataFrame(records)
        
        # Convert ISO3 codes to country names
        df['country'] = coco.convert(df['country'].tolist(), to='name_short', not_found=np.nan)
        df = df.dropna(subset=['country'])
        
        print(f"   ✅ Found {len(df):,} records")
        return df
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        return pd.DataFrame()

# Fetch UNICEF population data
population_df = fetch_unicef_under5_population()
population_df['indicator'] = 'Population ages 0-4 (number)'
population_df['source'] = 'UNICEF'

print("\n📊 Sample population data:")
population_df.head(10)


📥 Fetching: Under-5 population from UNICEF...


WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex
WORLD not found in regex


   ✅ Found 52,392 records

📊 Sample population data:


,country,year,value,indicator,source
0,Burundi,1950,378599.0,Population ages 0-4 (number),UNICEF
1,Burundi,1951,401022.0,Population ages 0-4 (number),UNICEF
2,Burundi,1952,421930.0,Population ages 0-4 (number),UNICEF
3,Burundi,1953,441579.0,Population ages 0-4 (number),UNICEF
4,Burundi,1954,461923.0,Population ages 0-4 (number),UNICEF
5,Burundi,1955,475491.0,Population ages 0-4 (number),UNICEF
6,Burundi,1956,480900.0,Population ages 0-4 (number),UNICEF
7,Burundi,1957,486567.0,Population ages 0-4 (number),UNICEF
8,Burundi,1958,492461.0,Population ages 0-4 (number),UNICEF
9,Burundi,1959,498806.0,Population ages 0-4 (number),UNICEF


---

## 🔄 Step 6: Consolidate All Data Sources

Now let's combine all the data we've fetched into a single, clean dataset.


In [ ]:
# Combine all data sources
all_data = pd.concat([wb_combined, who_combined, population_df], ignore_index=True)

def delist(x):
    if type(x) == list:
        return ' '.join(x)
    else:
        return x

all_data['country'] = all_data['country'].apply(delist)
print("📊 Combined Dataset Summary:")
print(f"   Total records: {len(all_data):,}")
print(f"   Unique countries: {all_data['country'].nunique()}")
print(f"   Indicators: {all_data['indicator'].nunique()}")
print(f"   Year range: {all_data['year'].min()} - {all_data['year'].max()}")

print("\n📋 Records by indicator:")
all_data.groupby('indicator').size().sort_values(ascending=False)


📊 Combined Dataset Summary:
   Total records: 145,940
   Unique countries: 302
   Indicators: 7
   Year range: 1950 - 2024

📋 Records by indicator:


indicator
Wasting prevalence (children under 5)         67739
Population ages 0-4 (number)                  52392
Under-5 mortality rate (per 1000)             13251
Solid fuel use (%)                             6521
Low birth weight (≤2500g)                      3953
Malnutrition (weight-for-age < -2 SD)          1477
Exclusive breastfeeding under 6 months (%)      607
dtype: int64

In [33]:
# Get the latest value for each country-indicator combination
latest_values = (
    all_data
    .sort_values('year', ascending=False)
    .drop_duplicates(['country', 'indicator'])
)

# Pivot to wide format (countries as rows, indicators as columns)
summary_table = latest_values.pivot(
    index='country',
    columns='indicator',
    values='value'
).reset_index()

summary_table.columns.name = None

print(f"📊 Summary table: {len(summary_table)} countries × {len(summary_table.columns)-1} indicators")
summary_table.head(15)


📊 Summary table: 302 countries × 7 indicators


,country,Exclusive breastfeeding under 6 months (%),Low birth weight (≤2500g),Malnutrition (weight-for-age < -2 SD),Population ages 0-4 (number),Solid fuel use (%),Under-5 mortality rate (per 1000),Wasting prevalence (children under 5)
0,Afghanistan,57.50,NaN,18.4,3400600.0,NaN,55.500000,2.3
1,Africa Eastern and Southern,NaN,14.282868,NaN,NaN,18.710000,53.806252,NaN
2,Africa Western and Central,NaN,NaN,NaN,NaN,1.818465,88.726335,NaN
3,Albania,36.54,5.992350,1.5,75218.0,36.810000,9.400000,1.9
4,Algeria,28.63,7.197213,2.7,2425767.0,0.090000,22.000000,2.7
5,American Samoa,NaN,NaN,NaN,3854.0,NaN,NaN,NaN
6,Andorra,NaN,9.394871,NaN,1260.0,NaN,2.600000,NaN
7,Angola,37.38,15.527791,19.0,3133180.0,6.550000,64.000000,5.7
8,Anguilla,NaN,NaN,NaN,349.0,NaN,NaN,NaN
9,Antigua and Barbuda,NaN,15.355503,NaN,2741.0,NaN,9.300000,NaN


---

## 💾 Step 7: Save Your Results

Let's save both the long-format data and the summary table to files.


In [34]:
import os


# Create output directory
output_dir = 'tutorial_output'
os.makedirs(output_dir, exist_ok=True)

# Save to Excel with multiple sheets
output_excel = f'{output_dir}/gbd_data_package.xlsx'

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    # Sheet 1: Summary table (wide format)
    summary_table.to_excel(writer, sheet_name='Summary', index=False)
    
    # Sheet 2: All data (long format)
    all_data.to_excel(writer, sheet_name='All_Data', index=False)
    
    # Sheet 3: Just population data
    population_df.to_excel(writer, sheet_name='Population', index=False)

print(f"✅ Excel file saved: {output_excel}")

# Also save as CSV
summary_csv = f'{output_dir}/gbd_summary.csv'
summary_table.to_csv(summary_csv, index=False)
print(f"✅ CSV file saved: {summary_csv}")


✅ Excel file saved: tutorial_output/gbd_data_package.xlsx
✅ CSV file saved: tutorial_output/gbd_summary.csv


---

## 🎉 Congratulations!

You've successfully:

1. ✅ Connected to the **World Bank API** and fetched malnutrition, birth weight, and mortality data
2. ✅ Connected to the **WHO GHO API** and fetched wasting and breastfeeding data
3. ✅ Connected to the **UNICEF SDMX API** and fetched under-5 population data
4. ✅ **Consolidated** all data sources into a single dataset
5. ✅ **Saved** the results to Excel and CSV files

### Next Steps

- 📊 **Visualize** the data using matplotlib or plotly
- 🔍 **Analyze** trends over time or compare regions
- 🧮 **Impute** missing values using regional averages
- 📝 **Explore** the full pipeline in `gbd_pipeline_final.py`

---

## 📚 Quick Reference: API Endpoints

| Source | Base URL | Format |
|--------|----------|--------|
| World Bank | `http://api.worldbank.org/v2/country/all/indicator/{CODE}?format=json` | JSON |
| WHO GHO | `https://ghoapi.azureedge.net/api/{CODE}` | JSON (OData) |
| UNICEF | `https://sdmx.data.unicef.org/ws/public/sdmxapi/rest/data/UNICEF,DM,1.0/.{CODE}?format=sdmx-json` | SDMX-JSON |

### Common Indicator Codes

**World Bank:**
- `SH.STA.MALN.ZS` - Malnutrition (weight-for-age)
- `SH.STA.BRTW.ZS` - Low birth weight
- `SH.DYN.MORT` - Under-5 mortality rate
- `EG.USE.COMM.CL.ZS` - Solid fuel use

**WHO:**
- `NUTRITION_WH_2` - Wasting prevalence
- `WHOSIS_000006` - Exclusive breastfeeding

**UNICEF:**
- `DM_POP_U5` - Population under 5


In [37]:
# Final summary
print("=" * 60)
print("📊 FINAL DATA PACKAGE SUMMARY")
print("=" * 60)
print(f"\n🌍 Countries covered: {len(summary_table)}")
print(f"📋 Indicators included: {len(summary_table.columns) - 1}")
print(f"📁 Output files:")
print(f"   • {output_excel}")
print(f"   • {summary_csv}")
print(f"\n⏱️ Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n✅ Tutorial complete! Happy analyzing! 🎉")


📊 FINAL DATA PACKAGE SUMMARY

🌍 Countries covered: 302
📋 Indicators included: 7
📁 Output files:
   • tutorial_output/gbd_data_package.xlsx
   • tutorial_output/gbd_summary.csv

⏱️ Completed: 2026-01-07 15:51:31

✅ Tutorial complete! Happy analyzing! 🎉
